# 2-1. 시계열 트렌드 분석 (Time Series EDA)

경기도 카드소비 x 날씨 데이터 EDA 2단계.
목표: 일별/월별 매출 추이, 요일 패턴, 계절 패턴을 확인하고 데이터 누락 구간이 있는지 점검한다.
분류등급(필수/애매/불필요)별 비교를 유지한다.

참고: 요일은 원본 `day` 코드(1~7, 정확한 매핑 불명) 대신 `date`(datetime) 컬럼에서
직접 계산한다 — 해석의 모호함이 없는 안전한 방법이다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2-1-0. 데이터 로드

In [ ]:
from pathlib import Path

# src/analysis/ 에서 실행하는 걸 기준으로 프로젝트 루트까지 2단계 상위 경로.
CATEGORY_DIR = Path("../../data/processed/consume_weather_by_category")

GROUP_ORDER = ["필수", "애매", "불필요"]

group_dfs = {}
for group in GROUP_ORDER:
    g_df = pd.read_parquet(CATEGORY_DIR / f"{group}.parquet")
    g_df["분류등급"] = group
    group_dfs[group] = g_df

df = pd.concat(group_dfs.values(), ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["요일"] = df["date"].dt.day_name()
df["요일_한글"] = df["date"].dt.dayofweek.map({0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"})
df["연월"] = df["date"].dt.to_period("M")

print("shape:", df.shape)
print("기간:", df["date"].min(), "~", df["date"].max())
df.head()

## 2-1-1. 일별 총매출 추이 (분류등급별)

카드 소비 일별 데이터는 요일 효과 때문에 노이즈가 크므로, 원본 라인과
7일 이동평균(rolling mean)을 같이 그려서 추세를 더 명확히 본다.

In [ ]:
daily = df.groupby(["date", "분류등급"])["매출금액"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for g in GROUP_ORDER:
    sub = daily[daily["분류등급"] == g].sort_values("date")
    ax.plot(sub["date"], sub["매출금액"], alpha=0.25, linewidth=0.8)
    ax.plot(sub["date"], sub["매출금액"].rolling(7, min_periods=1).mean(), label=f"{g} (7일 이동평균)", linewidth=1.8)

ax.set_title("일별 총매출 추이 (분류등급별, 옅은선=원본/굵은선=7일 이동평균)")
ax.set_xlabel("날짜")
ax.set_ylabel("매출금액 합계")
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2-1-2. 도시별 일별 추이 (필수 그룹 기준)

9개 도시를 한 그래프에 다 겹치면 알아보기 어려우므로, 분석 핵심인 '필수' 그룹만 도시별로 비교한다.

In [ ]:
essential = df[df["분류등급"] == "필수"]
daily_city = essential.groupby(["date", "도시"])["매출금액"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for city in sorted(daily_city["도시"].unique()):
    sub = daily_city[daily_city["도시"] == city].sort_values("date")
    ax.plot(sub["date"], sub["매출금액"].rolling(7, min_periods=1).mean(), label=city, linewidth=1.2)

ax.set_title("도시별 일별 매출금액 추이 (필수 그룹, 7일 이동평균)")
ax.set_xlabel("날짜")
ax.set_ylabel("매출금액 합계")
ax.legend(loc="upper left", ncol=3, fontsize=8)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2-1-3. 요일 패턴

평일/주말 소비 패턴 차이를 분류등급별로 비교한다.
매출금액은 왜도가 크므로 로그변환된 값으로 비교한다.

In [ ]:
weekday_order = ["월", "화", "수", "목", "금", "토", "일"]

weekday_stats = (
    df.groupby(["요일_한글", "분류등급"])["매출금액"]
    .apply(lambda x: np.log1p(x).mean())
    .reset_index()
    .pivot(index="요일_한글", columns="분류등급", values="매출금액")
    .reindex(weekday_order)
)

fig, ax = plt.subplots(figsize=(8, 5))
weekday_stats[GROUP_ORDER].plot(kind="bar", ax=ax)
ax.set_title("요일별 평균 log(매출금액+1) (분류등급별)")
ax.set_xlabel("요일")
ax.set_ylabel("평균 log(매출금액 + 1)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2-1-4. 월별(계절) 패턴

연도 효과를 제거하고 순수 계절성만 보기 위해, 연도-월이 아니라 '월'(1~12)만 기준으로
평균을 낸다 (2022~2025년 데이터를 월별로 합쳐서 봄).

In [ ]:
df["월"] = df["date"].dt.month

monthly_stats = (
    df.groupby(["월", "분류등급"])["매출금액"]
    .apply(lambda x: np.log1p(x).mean())
    .reset_index()
    .pivot(index="월", columns="분류등급", values="매출금액")
)

fig, ax = plt.subplots(figsize=(10, 5))
for g in GROUP_ORDER:
    ax.plot(monthly_stats.index, monthly_stats[g], marker="o", label=g)

ax.set_title("월별 평균 log(매출금액+1) (분류등급별, 2022~2025 통합)")
ax.set_xlabel("월")
ax.set_ylabel("평균 log(매출금액 + 1)")
ax.set_xticks(range(1, 13))
ax.legend()
plt.tight_layout()
plt.show()

## 2-1-5. 데이터 누락 구간 확인

도시별로 2022-01-01 ~ 2025-12-31 기간 중 실제 데이터가 있는 날짜가 몇 %인지,
연속으로 비는 구간(7일 이상)이 있는지 확인한다.

In [ ]:
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
print(f"전체 기간: {len(full_range)}일\n")

for city in sorted(df["도시"].unique()):
    city_dates = set(df.loc[df["도시"] == city, "date"].unique())
    missing_dates = sorted(set(full_range) - city_dates)
    coverage = (1 - len(missing_dates) / len(full_range)) * 100
    print(f"[{city}] 커버리지 {coverage:.1f}% (누락 {len(missing_dates)}일)")

    if missing_dates:
        # 연속 누락 구간(7일 이상)만 추려서 출력
        gaps = []
        start = missing_dates[0]
        prev = missing_dates[0]
        for d in missing_dates[1:]:
            if (d - prev).days > 1:
                gaps.append((start, prev))
                start = d
            prev = d
        gaps.append((start, prev))

        long_gaps = [(s, e) for s, e in gaps if (e - s).days + 1 >= 7]
        for s, e in long_gaps:
            print(f"    - {s.date()} ~ {e.date()} ({(e - s).days + 1}일 연속 누락)")

## 2단계 요약 (직접 채워넣기)

- 전체 추세: ...
- 요일 패턴: 평일/주말 차이가 [뚜렷함 / 미미함], 분류등급별 차이 ...
- 계절 패턴: [특정 월/계절]에 [증가/감소] 경향
- 데이터 누락: [있음 / 없음], 있다면 처리 방향 ...

다음 단계(3단계: 범주형 변수별 분해)로 이동.